In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.GoldHelper

In [0]:
from pyspark.sql import functions as F 

In [0]:
target_table=f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
circuit_bronze_table=f"{catalog_name}.{silver_schema}.circuits" 

In [0]:

races_bronze_table=f"{catalog_name}.{silver_schema}.races"

In [0]:

df_races = (spark.table(races_bronze_table)
            .filter(F.col("batch_id")==v_batch_id)
            )
df_circuits = (spark.table(circuit_bronze_table)
                .filter(F.col("batch_id")==v_batch_id)
            )


In [0]:
dim_races_df=(
    df_races
    .join(df_circuits,
    df_races.circuit_id==df_circuits.circuit_id,
    "inner")
    .select(
        df_races.season.alias("season"),
        df_races.round.alias("round"),
        df_circuits.circuit_name.alias("race_name"),
        df_races.race_date.alias("race_date"),
        df_circuits.circuit_id.alias("circuit_id"),
        df_circuits.circuit_name.alias("circuit_name"),
        df_circuits.locality.alias("locality"),
        df_circuits.country.alias("country")
    )
)

In [0]:
display(dim_races_df)

In [0]:
# (dim_races_df
#  .write
#  .format("delta")
#  .mode("overwrite")
#  .saveAsTable(target_table)
# )

In [0]:
write_to_gold(
    input_df=dim_races_df,
    target_table=target_table,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "locality",
        "country"
    ]
)

In [0]:
display(spark.table(target_table))